In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load and clean
df = pd.read_csv("cement_demand_features.csv")
target = "consumed_tonnes"

df = df.drop_duplicates()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=[target, "date"]).sort_values("date")
df = df.reset_index(drop=True)

# Create date features
df["date_year"] = df["date"].dt.year
df["date_month"] = df["date"].dt.month
df["date_day"] = df["date"].dt.day
df["date_dayofweek"] = df["date"].dt.dayofweek

# Keep target and exclude raw date
X = df.drop(columns=[target, "date"], errors="ignore")
y = df[target]

X = X.select_dtypes(exclude=["complex"])
X = X.dropna(axis=1, how="all")

# 2. Chronological split: earlier data trains, later data tests
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError("Not enough rows for a chronological train/test split.")

# 3. Identify feature types
numeric_cols = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

# 4. Preprocessing
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

# 5. Build model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# 6. Train and evaluate
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 4))
print("R2:", round(r2_score(y_test, pred), 4))

# 7. Save
os.makedirs("saved_models", exist_ok=True)

joblib.dump(model, "saved_models/cement_demand_model.pkl")
joblib.dump({
    "target": target,
    "features": X.columns.tolist(),
    "date_column": "date",
    "split_method": "chronological"
}, "saved_models/cement_demand_model_metadata.pkl")

print("Cement demand model and metadata saved.")

MAE: 4.1188
RMSE: 7.2501
R2: 0.8155
Cement demand model and metadata saved.
